# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

In [17]:
import json
import os
import re
import random
import numpy as np
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "data"

def load_json(path: str) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_claims           = load_json(os.path.join(DATA_DIR, "train-claims.json"))
dev_claims             = load_json(os.path.join(DATA_DIR, "dev-claims.json"))
test_claims_unlabelled = load_json(os.path.join(DATA_DIR, "test-claims-unlabelled.json"))
evidence               = load_json(os.path.join(DATA_DIR, "evidence.json"))
dev_baseline           = load_json(os.path.join(DATA_DIR, "dev-claims-baseline.json"))

print("=== Dataset Overview ===")
print(f"  Train claims             : {len(train_claims):>8,}")
print(f"  Dev   claims             : {len(dev_claims):>8,}")
print(f"  Test  claims (unlabelled): {len(test_claims_unlabelled):>8,}")
print(f"  Evidence passages        : {len(evidence):>8,}")
print(f"  Baseline predictions     : {len(dev_baseline):>8,}")


=== Dataset Overview ===
  Train claims             :    1,228
  Dev   claims             :      154
  Test  claims (unlabelled):      153
  Evidence passages        : 1,208,827
  Baseline predictions     :      154


In [ ]:
# ── Label space ───────────────────────────────────────────────────────────────
LABELS   = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}

print("=== Claim Label Space ===")
for label, idx in label2id.items():
    print(f"  [{idx}] {label}")

# ── Show one labelled claim and its evidence passages (matches README format) ──
sample_cid   = list(train_claims.keys())[0]
sample_claim = train_claims[sample_cid]

print(f"\n=== Example Labelled Claim (train-claims.json) ===")
print(f"  Claim ID   : {sample_cid}")
print(f"  Claim Text : {sample_claim['claim_text']}")
print(f"  Label      : {sample_claim['claim_label']}")
print(f"  Evidence   : {sample_claim['evidences']}")
print("\n  --- Corresponding Evidence Passages ---")
for eid in sample_claim["evidences"]:
    text = evidence[eid]
    print(f"  [{eid}]: {text[:120]}{'...' if len(text) > 120 else ''}")

# ── Show one unlabelled test claim ────────────────────────────────────────────
test_cid   = list(test_claims_unlabelled.keys())[0]
test_claim = test_claims_unlabelled[test_cid]

print(f"\n=== Example Unlabelled Test Claim (test-claims-unlabelled.json) ===")
print(f"  Claim ID   : {test_cid}")
print(f"  Claim Text : {test_claim['claim_text']}")
print("  (No label or evidences — system must predict both)")

# ── Show baseline output format (this is what our predictions must look like) ─
baseline_cid  = list(dev_baseline.keys())[0]
baseline_pred = dev_baseline[baseline_cid]

print(f"\n=== Example Baseline Prediction (dev-claims-baseline.json) ===")
print(f"  Claim ID   : {baseline_cid}")
print(f"  Predicted Label    : {baseline_pred.get('claim_label')}")
print(f"  Predicted Evidences: {baseline_pred.get('evidences', [])[:5]}")


=== Claim Label Space ===
  [0] SUPPORTS
  [1] REFUTES
  [2] NOT_ENOUGH_INFO
  [3] DISPUTED

=== Example Labelled Claim (train-claims.json) ===
  Claim ID   : claim-1937
  Claim Text : Not only is there no scientific evidence that CO2 is a pollutant, higher CO2 concentrations actually help ecosystems support more plant and animal life.
  Label      : DISPUTED
  Evidence   : ['evidence-442946', 'evidence-1194317', 'evidence-12171']

  --- Corresponding Evidence Passages ---
  [evidence-442946]: At very high concentrations (100 times atmospheric concentration, or greater), carbon dioxide can be toxic to animal lif...
  [evidence-1194317]: Plants can grow as much as 50 percent faster in concentrations of 1,000 ppm CO 2 when compared with ambient conditions, ...
  [evidence-12171]: Higher carbon dioxide concentrations will favourably affect plant growth and demand for water.

=== Example Unlabelled Test Claim (test-claims-unlabelled.json) ===
  Claim ID   : claim-2967
  Claim Text : The co

In [ ]:
# ── (A) Label distribution ────────────────────────────────────────────────────
print("=== (A) Label Distribution ===")
print(f"{'Label':<22} {'Train':>6}  {'Dev':>6}  {'Train%':>7}")
print("-" * 48)
train_label_counts = Counter(v["claim_label"] for v in train_claims.values())
dev_label_counts   = Counter(v["claim_label"] for v in dev_claims.values())
total_train        = sum(train_label_counts.values())
for label in LABELS:
    tr = train_label_counts[label]
    dv = dev_label_counts.get(label, 0)
    print(f"  {label:<20} {tr:>6,}  {dv:>6,}  {100*tr/total_train:>6.1f}%")
print(f"  {'TOTAL':<20} {total_train:>6,}  {len(dev_claims):>6,}")

# Class imbalance note: SUPPORTS dominates → affects classification baseline
majority_label = train_label_counts.most_common(1)[0][0]
print(f"\n  Majority class (train): {majority_label} — a classifier predicting this "
      f"always scores {100*train_label_counts[majority_label]/total_train:.1f}% accuracy")

# ── (B) Claim text length (words) ─────────────────────────────────────────────
print("\n=== (B) Claim Text Length (words) ===")
train_lengths = [len(v["claim_text"].split()) for v in train_claims.values()]
dev_lengths   = [len(v["claim_text"].split()) for v in dev_claims.values()]
print(f"  {'Split':<8} {'Min':>5} {'Max':>5} {'Mean':>7} {'Median':>8}")
print(f"  {'Train':<8} {min(train_lengths):>5} {max(train_lengths):>5} "
      f"{np.mean(train_lengths):>7.1f} {np.median(train_lengths):>8.1f}")
print(f"  {'Dev':<8} {min(dev_lengths):>5} {max(dev_lengths):>5} "
      f"{np.mean(dev_lengths):>7.1f} {np.median(dev_lengths):>8.1f}")

# ── (C) Gold evidence passages per claim ─────────────────────────────────────
print("\n=== (C) Gold Evidence Passages per Claim ===")
gold_ev_counts = [len(v["evidences"]) for v in train_claims.values()]
ev_counter     = Counter(gold_ev_counts)
print(f"  Min={min(gold_ev_counts)}  Max={max(gold_ev_counts)}  "
      f"Mean={np.mean(gold_ev_counts):.2f}")
print("  Distribution:")
for n in sorted(ev_counter):
    print(f"    {n} evidence(s): {ev_counter[n]:4d} claims")

# ── (D) Evidence corpus statistics ───────────────────────────────────────────
print("\n=== (D) Evidence Corpus Statistics ===")
ev_lengths = [len(v.split()) for v in evidence.values()]
print(f"  Total passages : {len(ev_lengths):,}")
print(f"  Min length     : {min(ev_lengths)} words")
print(f"  Max length     : {max(ev_lengths)} words")
print(f"  Mean length    : {np.mean(ev_lengths):.1f} words")
print(f"  Median length  : {np.median(ev_lengths):.1f} words")

# ── (E) Baseline evaluation metrics (from README §3) ─────────────────────────
# README reports: F=0.3378, A=0.3506, H=0.3441 for baseline on dev set
# Our goal is to exceed these metrics with a principled retrieval+classification system
print("\n=== (E) Reference: Baseline Metrics (from README §3) ===")
print("  Evidence Retrieval F-score (F) = 0.3378")
print("  Claim Classification Accuracy (A) = 0.3506")
print("  Harmonic Mean of F and A  (H) = 0.3441")
print("  (Note: baseline uses randomly mixed evidence passages + random labels)")


=== (A) Label Distribution ===
Label                   Train     Dev   Train%
------------------------------------------------
  SUPPORTS                519      68    42.3%
  REFUTES                 199      27    16.2%
  NOT_ENOUGH_INFO         386      41    31.4%
  DISPUTED                124      18    10.1%
  TOTAL                 1,228     154

  Majority class (train): SUPPORTS — a classifier predicting this always scores 42.3% accuracy

=== (B) Claim Text Length (words) ===
  Split      Min   Max    Mean   Median
  Train        4    67    20.1     19.0
  Dev          4    65    21.1     18.0

=== (C) Gold Evidence Passages per Claim ===
  Min=1  Max=5  Mean=3.36
  Distribution:
    1 evidence(s):  210 claims
    2 evidence(s):  223 claims
    3 evidence(s):  191 claims
    4 evidence(s):  127 claims
    5 evidence(s):  477 claims

=== (D) Evidence Corpus Statistics ===
  Total passages : 1,208,827
  Min length     : 1 words
  Max length     : 479 words
  Mean length    : 19.7 

In [ ]:
def clean_text(text: str) -> str:
    """Normalise whitespace; preserve all vocabulary for climate-domain retrieval."""
    text = re.sub(r"[\r\n\t]+", " ", text)  # collapse newlines / tabs
    text = re.sub(r"\s{2,}", " ", text)      # collapse repeated spaces
    return text.strip()

# Build parallel lists: evidence_ids[i]  ↔  evidence_texts[i]
# This positional alignment is required by the TF-IDF matrix (rows = passages).
evidence_ids   = list(evidence.keys())
evidence_texts = [clean_text(evidence[eid]) for eid in evidence_ids]

# Sanity check
empty_count = sum(1 for t in evidence_texts if len(t) == 0)
print(f"Evidence passages after cleaning : {len(evidence_texts):,}")
print(f"Empty passages                   : {empty_count}")
print(f"\nSample passage [{evidence_ids[0]}]:")
print(f"  {evidence_texts[0][:200]}{'...' if len(evidence_texts[0]) > 200 else ''}")


Evidence passages after cleaning : 1,208,827
Empty passages                   : 0

Sample passage [evidence-0]:
  John Bennet Lawes, English entrepreneur and agricultural scientist


In [ ]:
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# ── Global hyperparameters (shared across Sections 1-3) ──────────────────────
MODEL_NAME   = "bert-base-uncased"       # BERT classifier backbone
SBERT_MODEL  = "multi-qa-MiniLM-L6-cos-v1"  # sentence encoder for re-ranking
TOP_K        = 5      # final evidence passages returned per claim
COARSE_K     = 100    # TF-IDF candidate pool before SBERT re-ranking
ALPHA        = 0.2    # weight for TF-IDF in hybrid score (SBERT weight = 1-ALPHA)
MAX_LENGTH   = 384    # BERT tokeniser max sequence length
BATCH_SIZE   = 8
NUM_EPOCHS   = 4
LR           = 2e-5
WARMUP_RATIO = 0.10
CKPT_PATH    = "best_bert_classifier.pt"

DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL_MAP = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL  = {idx: label for label, idx in LABEL_MAP.items()}

print(f"Device            : {DEVICE}")
print(f"TOP_K / COARSE_K  : {TOP_K} / {COARSE_K}")
print(f"Hybrid alpha      : {ALPHA} (TF-IDF) / {1-ALPHA} (SBERT)")

# ── Stage 1: TF-IDF sparse index ──────────────────────────────────────────────
print("\nBuilding TF-IDF index …")
tfidf = TfidfVectorizer(
    max_features=500_000,  # cap vocabulary; balances memory vs. recall
    sublinear_tf=True,     # log-normalise term frequency (1 + log tf)
    analyzer="word",
    ngram_range=(1, 1),    # unigrams only; bigrams add noise without clear gain
    min_df=2,              # discard terms appearing in only 1 passage (noise)
)
tfidf_matrix = tfidf.fit_transform(evidence_texts)
print(f"TF-IDF matrix     : {tfidf_matrix.shape}  "
      f"(rows=passages, cols=vocab, nnz={tfidf_matrix.nnz:,})")

# ── Stage 2: SBERT semantic encoder ───────────────────────────────────────────
print("\nLoading SBERT encoder …")
sbert = SentenceTransformer(SBERT_MODEL)
print("SBERT loaded successfully.")


Device            : cpu
TOP_K / COARSE_K  : 5 / 100
Hybrid alpha      : 0.2 (TF-IDF) / 0.8 (SBERT)

Building TF-IDF index …
TF-IDF matrix     : (1208827, 233783)  (rows=passages, cols=vocab, nnz=19,928,857)

Loading SBERT encoder …


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SBERT loaded successfully.


# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*